# 03 tmdb revenue collection

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

print("=" * 60)
print("GOOGLE DRIVE PROJECT CHECK")
print("=" * 60)

print("Project exists:", os.path.exists(PROJECT_ROOT))

if os.path.exists(PROJECT_ROOT):
    print("\nProject contents:")
    print(os.listdir(PROJECT_ROOT))

DATA_DIR = os.path.join(PROJECT_ROOT, "01_data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
MAPPING_PATH = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_TMDB_MAPPING.csv"
)

print("\n01_data exists:", os.path.exists(DATA_DIR))
print("processed exists:", os.path.exists(PROCESSED_DIR))
print("Mapping exists:", os.path.exists(MAPPING_PATH))

if os.path.exists(PROCESSED_DIR):
    print("\nProcessed files:")
    print(os.listdir(PROCESSED_DIR))

In [ ]:
import pandas as pd
import os

# ============================================================
# LOAD VERIFIED TMDB MAPPING
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

INPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_TMDB_MAPPING.csv"
)

print("=" * 60)
print("LOADING VERIFIED TMDB MAPPING")
print("=" * 60)

print("Input:", INPUT_PATH)
print("Input exists:", os.path.exists(INPUT_PATH))

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"TMDB mapping file not found:\n{INPUT_PATH}"
    )

tmdb_mapping = pd.read_csv(INPUT_PATH)

print("\nTMDB mapping loaded successfully.")
print("Rows:", len(tmdb_mapping))
print("Columns:", len(tmdb_mapping.columns))

# Validation
print("\nValidation:")
print("Unique movie records:",
      tmdb_mapping["movie_record_id"].nunique())

print("Unique TMDB IDs:",
      tmdb_mapping["tmdb_id"].nunique())

print("Missing TMDB IDs:",
      tmdb_mapping["tmdb_id"].isna().sum())

print("Duplicate movie records:",
      tmdb_mapping["movie_record_id"].duplicated().sum())

print("Duplicate TMDB IDs:",
      tmdb_mapping["tmdb_id"].duplicated().sum())

print("\nFirst 5 records:")
display(
    tmdb_mapping[
        [
            "movie_record_id",
            "dataset_title",
            "dataset_year",
            "dataset_language",
            "tmdb_id",
            "tmdb_title"
        ]
    ].head()
)

In [ ]:
# ============================================================
# LOAD TMDB API CONFIGURATION
# ============================================================

import os
import sys
import importlib.util

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
CONFIG_FILE = os.path.join(
    PROJECT_ROOT,
    "04_config",
    "config.py"
)

print("=" * 60)
print("LOADING TMDB CONFIGURATION")
print("=" * 60)

print("Config file:", CONFIG_FILE)
print("Config exists:", os.path.exists(CONFIG_FILE))

if not os.path.exists(CONFIG_FILE):
    raise FileNotFoundError(
        f"config.py not found at:\n{CONFIG_FILE}"
    )

# Load config.py directly from its absolute path
spec = importlib.util.spec_from_file_location(
    "project_config",
    CONFIG_FILE
)

config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(config)

# Read configuration values
TMDB_API_KEY = config.TMDB_API_KEY
BASE_URL = config.BASE_URL
LANGUAGE = config.LANGUAGE
REQUEST_DELAY = config.REQUEST_DELAY
MAX_RETRIES = config.MAX_RETRIES
TIMEOUT = config.TIMEOUT

print("\nConfiguration loaded successfully.")
print("TMDB API key loaded:", bool(TMDB_API_KEY))
print("Base URL:", BASE_URL)
print("Language:", LANGUAGE)
print("Request delay:", REQUEST_DELAY)
print("Max retries:", MAX_RETRIES)
print("Timeout:", TIMEOUT)

In [ ]:
# ============================================================
# TMDB REVENUE COLLECTION — SETUP
# ============================================================

import os
import time
import requests
import pandas as pd

print("=" * 60)
print("TMDB REVENUE COLLECTION — SETUP")
print("=" * 60)

# TMDB movie endpoint
MOVIE_ENDPOINT = f"{BASE_URL}/movie"

print("Movie endpoint:", MOVIE_ENDPOINT)
print("Movies to process:", len(tmdb_mapping))

# Basic validation
if len(tmdb_mapping) != 130:
    raise ValueError(
        f"Expected 130 movies, found {len(tmdb_mapping)}"
    )

if tmdb_mapping["tmdb_id"].isna().any():
    raise ValueError("Missing TMDB IDs detected.")

if tmdb_mapping["tmdb_id"].duplicated().any():
    raise ValueError("Duplicate TMDB IDs detected.")

print("\n✅ Revenue collection setup validated.")

In [ ]:
# ============================================================
# TMDB REVENUE API — SINGLE MOVIE TEST
# ============================================================

test_tmdb_id = int(tmdb_mapping.iloc[0]["tmdb_id"])
test_title = tmdb_mapping.iloc[0]["dataset_title"]

url = f"{MOVIE_ENDPOINT}/{test_tmdb_id}"

params = {
    "api_key": TMDB_API_KEY,
    "language": LANGUAGE
}

response = requests.get(
    url,
    params=params,
    timeout=TIMEOUT
)

print("=" * 60)
print("TMDB REVENUE API TEST")
print("=" * 60)

print("Movie:", test_title)
print("TMDB ID:", test_tmdb_id)
print("HTTP Status:", response.status_code)

if response.status_code != 200:
    print("\n❌ TMDB request failed")
    print(response.text[:1000])
    raise RuntimeError("TMDB revenue test failed.")

data = response.json()

print("\nTMDB response successful.")
print("Title:", data.get("title"))
print("Release date:", data.get("release_date"))
print("Budget:", data.get("budget"))
print("Revenue:", data.get("revenue"))
print("Runtime:", data.get("runtime"))

print("\n✅ Single-movie revenue test PASSED.")

In [ ]:
# ============================================================
# TMDB FINANCIAL METADATA COLLECTION — 130 MOVIES
# ============================================================

import os
import time
import requests
import pandas as pd

financial_records = []
failed_requests = []

total_movies = len(tmdb_mapping)

print("=" * 60)
print("TMDB FINANCIAL METADATA COLLECTION")
print("=" * 60)
print(f"Starting collection for {total_movies} movies...\n")

for i, row in tmdb_mapping.iterrows():

    movie_record_id = row["movie_record_id"]
    tmdb_id = int(row["tmdb_id"])
    dataset_title = row["dataset_title"]

    url = f"{MOVIE_ENDPOINT}/{tmdb_id}"

    params = {
        "api_key": TMDB_API_KEY,
        "language": LANGUAGE
    }

    success = False

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=TIMEOUT
            )

            if response.status_code == 200:

                data = response.json()

                financial_records.append({
                    "movie_record_id": movie_record_id,
                    "tmdb_id": tmdb_id,
                    "dataset_title": dataset_title,
                    "tmdb_title": data.get("title"),
                    "budget": data.get("budget"),
                    "revenue": data.get("revenue"),
                    "release_date": data.get("release_date"),
                    "runtime": data.get("runtime")
                })

                success = True
                break

            elif response.status_code == 429:
                print(
                    f"[{i+1}/{total_movies}] Rate limited. "
                    f"Waiting before retry..."
                )
                time.sleep(2 ** attempt)

            else:
                print(
                    f"[{i+1}/{total_movies}] HTTP "
                    f"{response.status_code}: {dataset_title}"
                )

        except requests.exceptions.RequestException as e:

            print(
                f"[{i+1}/{total_movies}] Attempt "
                f"{attempt} failed: {dataset_title}"
            )

            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)

    if not success:

        failed_requests.append({
            "movie_record_id": movie_record_id,
            "tmdb_id": tmdb_id,
            "dataset_title": dataset_title
        })

    print(
        f"[{i+1}/{total_movies}] "
        f"{dataset_title} → "
        f"{'OK' if success else 'FAILED'}"
    )

    time.sleep(REQUEST_DELAY)

# Convert to DataFrame
tmdb_financials = pd.DataFrame(financial_records)

print("\n" + "=" * 60)
print("TMDB FINANCIAL COLLECTION COMPLETE")
print("=" * 60)

print("Expected movies:", total_movies)
print("Successfully collected:", len(tmdb_financials))
print("Failed:", len(failed_requests))

In [ ]:
# ============================================================
# VALIDATE TMDB FINANCIAL METADATA
# ============================================================

print("=" * 60)
print("TMDB FINANCIAL METADATA VALIDATION")
print("=" * 60)

print("Rows:", len(tmdb_financials))
print(
    "Unique movie_record_id:",
    tmdb_financials["movie_record_id"].nunique()
)
print(
    "Unique tmdb_id:",
    tmdb_financials["tmdb_id"].nunique()
)

print(
    "Missing movie_record_id:",
    tmdb_financials["movie_record_id"].isna().sum()
)

print(
    "Missing tmdb_id:",
    tmdb_financials["tmdb_id"].isna().sum()
)

print(
    "Duplicate movie_record_id:",
    tmdb_financials["movie_record_id"].duplicated().sum()
)

print(
    "Duplicate tmdb_id:",
    tmdb_financials["tmdb_id"].duplicated().sum()
)

# Compare against verified mapping
expected_ids = set(tmdb_mapping["movie_record_id"])
collected_ids = set(tmdb_financials["movie_record_id"])

missing_ids = expected_ids - collected_ids
extra_ids = collected_ids - expected_ids

print("\nExpected movie records:", len(expected_ids))
print("Collected movie records:", len(collected_ids))

print("Missing movie records:", missing_ids)
print("Extra movie records:", extra_ids)

# Final validation
if (
    len(tmdb_financials) == 130
    and tmdb_financials["movie_record_id"].nunique() == 130
    and tmdb_financials["tmdb_id"].nunique() == 130
    and tmdb_financials["tmdb_id"].isna().sum() == 0
    and tmdb_financials["movie_record_id"].duplicated().sum() == 0
    and tmdb_financials["tmdb_id"].duplicated().sum() == 0
    and len(missing_ids) == 0
    and len(extra_ids) == 0
):
    print("\n✅ TMDB FINANCIAL METADATA VALIDATION PASSED")
else:
    print("\n❌ VALIDATION FAILED — DO NOT SAVE YET")

In [ ]:
# ============================================================
# SAVE TMDB FINANCIAL METADATA
# ============================================================

import os

RAW_DIR = os.path.join(PROJECT_ROOT, "01_data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    RAW_DIR,
    "ICMB_130_TMDB_FINANCIALS.csv"
)

tmdb_financials.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 60)
print("TMDB FINANCIAL METADATA SAVED")
print("=" * 60)

print("Output:", OUTPUT_PATH)
print("File exists:", os.path.exists(OUTPUT_PATH))
print("Rows saved:", len(tmdb_financials))
print("Columns saved:", len(tmdb_financials.columns))